In [1]:
import sys
sys.path.append("..")

In [2]:
import tqdm
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import cvxpy as cp
from copy import deepcopy

from src.data import *
from src.model import *
from src.recourse import *
from src.utils import *

warnings.filterwarnings('ignore')

In [3]:
def append_result(d, algorithm, seed, alpha, lamb, i, x_0, theta_0, x_r, theta_r=None):
    d["algorithm"].append(algorithm)
    d["seed"].append(seed)
    d["alpha"].append(alpha)
    d["lambda"].append(lamb)
    d["i"].append(i)
    d["x_0"].append(x_0.round(4))
    d["x_r"].append(x_r.round(4))
    d["theta_0"].append(theta_0.round(4))

In [19]:
def recourse_runner(seed: int, X: np.ndarray, recourse: Recourse, params: dict, dataset: Dataset):
    alpha = params['alpha']
    lamb = params['lamb']
    
    results = {'algorithm': [], 'seed': [], 'alpha': [], 'lambda': [], 'i': [], 'x_0': [], 'x_r': [], 'theta_0': []}
    weights_0, bias_0 = recourse.weights, recourse.bias
    theta_0 = np.hstack((weights_0, bias_0))
    if recourse.name == "ROAR":
        print(weights_0, bias_0, theta_0)
    n = len(X)
    for i in tqdm.trange(n, desc=f'[{recourse.name}] [alpha={alpha}] [lambda={lamb}]', colour='#0091ff'):
        x_0 = X[i]
        x_r = recourse.get_recourse(x_0)
        append_result(results, recourse.name, seed, alpha, lamb, i, x_0, theta_0, x_r)

    df_results = pd.DataFrame(results)
    if params["save_results"]:
        print(f'[{recourse.name}] Saving results for {dataset.name} run {seed}')
        df_results.to_pickle(f'../results/recourse/lr_{dataset.name}_{recourse.name}_{seed}.pkl')
    
    return df_results

In [20]:
def run_experiment(dataset: Dataset, recourse_fns: List[Recourse], params: dict, results: List):
    alpha = params['alpha']
    
    for seed in params['seeds']:
        train_data, test_data = dataset.get_data(seed)
        X_train, y_train = train_data
        X_test, y_test = test_data
        
        base_model = LR()
        base_model.train(X_train.values, y_train.values)
        
        weights_0 = base_model.model.coef_[0]
        bias_0 = base_model.model.intercept_
        
        recourse_needed_X_train = recourse_needed(base_model.predict, X_train.values)
        recourse_needed_X_test = recourse_needed(base_model.predict, X_test.values)
        
        for recourse_fn in recourse_fns:
            recourse = recourse_fn(weights=weights_0, bias=bias_0, alpha=alpha)
            if params["lamb"] is None:
                params['lamb'] = recourse.choose_lambda(recourse_needed_X_train, base_model.predict, X_train.values)
                recourse.lamb = params['lamb']
            
            df_results = recourse_runner(seed, recourse_needed_X_test, recourse, params, dataset)
            results.append(df_results)

In [24]:
torch.manual_seed(0)

d_results = {}
params = {}
params['alpha'] = 0.5 # float, None
params['lamb'] = 0.1
params['seeds'] = range(1)
params['save_results'] = True

datasets = [SBADataset()]
recourse_fns = [ROAR]

for dataset in datasets:
    results = []
    print(f'Running {dataset.name} data...')
    run_experiment(dataset, recourse_fns, params, results)
    
    d_results[dataset.name] = pd.concat(results)
    print(f'Finished {dataset.name}\n')

Running sba data...
tensor([ 0.3521, -0.2441,  0.0460,  0.1365,  0.3101,  0.1158,  0.1984, -0.1196,
        -0.0705,  0.1364, -0.0203, -3.2871, -0.0403,  0.0893, -1.3327,  0.0810,
         0.1078, -0.1492,  0.3101,  0.2589,  0.0054, -0.1959,  0.1306,  0.1332,
         0.0334]) tensor([5.0410]) [ 0.3521394  -0.24412505  0.04598925  0.13652857  0.31008685  0.11578908
  0.19844052 -0.11962943 -0.07045412  0.13641363 -0.0203308  -3.2870512
 -0.04032794  0.08934175 -1.3327265   0.08099625  0.10780956 -0.14923906
  0.31008685  0.25887677  0.00538959 -0.19592789  0.13059658  0.13316037
  0.0333982   5.0410438 ]


[ROAR] [alpha=0.5] [lambda=0.1]:   3%|▎         | 1/39 [00:15<09:36, 15.18s/it]


KeyboardInterrupt: 

In [22]:
d_results['synthetic']

,algorithm,seed,alpha,lambda,i,x_0,x_r,theta_0
0,Alg1,0,0.5,0.1,0,"[-2.3862, -1.7774]","[0.0, 2.085]","[1.9661, 1.9713, 0.0506]"
1,Alg1,0,0.5,0.1,1,"[-2.7105, -1.7402]","[0.0, 2.085]","[1.9661, 1.9713, 0.0506]"
2,Alg1,0,0.5,0.1,2,"[-2.3817, -1.9247]","[0.0, 2.085]","[1.9661, 1.9713, 0.0506]"
3,Alg1,0,0.5,0.1,3,"[-2.337, -1.0178]","[0.0, 2.085]","[1.9661, 1.9713, 0.0506]"
4,Alg1,0,0.5,0.1,4,"[-2.402, -2.7282]","[0.0, 2.085]","[1.9661, 1.9713, 0.0506]"
...,...,...,...,...,...,...,...,...
91,Alg1,0,0.5,0.1,91,"[-1.7741, -2.2538]","[0.0, 2.085]","[1.9661, 1.9713, 0.0506]"
92,Alg1,0,0.5,0.1,92,"[-2.2471, -3.4363]","[0.0, 2.085]","[1.9661, 1.9713, 0.0506]"
93,Alg1,0,0.5,0.1,93,"[-1.2901, -2.4369]","[0.0, 2.085]","[1.9661, 1.9713, 0.0506]"
94,Alg1,0,0.5,0.1,94,"[-2.911, -1.7206]","[0.0, 2.085]","[1.9661, 1.9713, 0.0506]"


In [ ]:
df = d_results["synthetic"]
df[df["algorithm"]=="Alg1"]

,algorithm,seed,alpha,lambda,i,x_0,x_r,theta_0
0,Alg1,0,0.5,0.1,0,"[-2.3862, -1.7774]","[1.9661, 1.9713, 0.0506]","[0.0, 2.085]"
1,Alg1,0,0.5,0.1,1,"[-2.7105, -1.7402]","[1.9661, 1.9713, 0.0506]","[0.0, 2.085]"
2,Alg1,0,0.5,0.1,2,"[-2.3817, -1.9247]","[1.9661, 1.9713, 0.0506]","[0.0, 2.085]"
3,Alg1,0,0.5,0.1,3,"[-2.337, -1.0178]","[1.9661, 1.9713, 0.0506]","[0.0, 2.085]"
4,Alg1,0,0.5,0.1,4,"[-2.402, -2.7282]","[1.9661, 1.9713, 0.0506]","[0.0, 2.085]"
...,...,...,...,...,...,...,...,...
91,Alg1,0,0.5,0.1,91,"[-1.7741, -2.2538]","[1.9661, 1.9713, 0.0506]","[0.0, 2.085]"
92,Alg1,0,0.5,0.1,92,"[-2.2471, -3.4363]","[1.9661, 1.9713, 0.0506]","[0.0, 2.085]"
93,Alg1,0,0.5,0.1,93,"[-1.2901, -2.4369]","[1.9661, 1.9713, 0.0506]","[0.0, 2.085]"
94,Alg1,0,0.5,0.1,94,"[-2.911, -1.7206]","[1.9661, 1.9713, 0.0506]","[0.0, 2.085]"
